In [1]:
import os
import string
import annoy
import codecs
import fileinput

from pymorphy2 import MorphAnalyzer
from stop_words import get_stop_words
from gensim.models import Word2Vec

import numpy as np
from tqdm.notebook import tqdm
import pandas as pd

Предобработаем ответы mail.ru из файла: к каждому вопросу присоединим 1 ответ и запишем в файл на будущее. Это позволит нам сэкономить время и ресурсы при дальнейшем препроцессинге текста

In [ ]:
n=0
lines = list()
with open('Otvety.txt', encoding='utf-8') as file:
    for line in file:
        lines.append(line)
        n += 1
        if n==10000:
            break

lines[:20]

['\n',
 '---\n',
 'вопрос о ТДВ)) давно и хорошо отдыхаем)) ЛИЧНО ВАМ здесь кого советовали завести?)) . \n',
 'хомячка.... \n',
 'мужика, йопаря, собачку и 50 кошек))). \n',
 'Общение !). \n',
 'паучка. \n',
 'Да пол мне бы памыть!<br>А таг то ни чо. Типа ни каво!. \n',
 'я тут вообще что бы пообщаться..... \n',
 'А мне советовали сиси завести...))). \n',
 'Ну, слава богу, мужика завести ещё не советовали))) А вот сватать к кому только не сватали). \n',
 'мне тут советовали завести любовника, мужа и много кошек )))<br>приветик ))). \n',
 '---\n',
 'Как парни относятся к цветным линзам? Если у девушки то зеленые глаза, то голубые...)) . \n',
 'меня вобще прикалывает эта тема :). \n',
 'когда этобыло редкость - было забавно, а когда все знают, что эта фальшивка, то уже не прикольно, как силиконовые сиськи или как налепленные синтетические волосы. \n',
 '---\n',
 'Что делать, сегодня нашёл 2 миллиона рублей? . \n',
 'Если это "счастье " действительно на вас свалилось, лучше пойти в милиц

In [16]:
lines.count('---\n')

1457

In [17]:
n=0
lines = list()
with open('prepared_answers.txt', encoding='utf-8') as file:
    for line in file:
        lines.append(line)
        n += 1
        if n==10000:
            break

lines[:20]

['\tвопрос о ТДВ)) давно и хорошо отдыхаем)) ЛИЧНО ВАМ здесь кого советовали завести?)) . \n',
 'Как парни относятся к цветным линзам? Если у девушки то зеленые глаза, то голубые...)) .\tменя вобще прикалывает эта тема :). \n',
 'Что делать, сегодня нашёл 2 миллиона рублей? .\tЕсли это "счастье " действительно на вас свалилось, лучше пойти в милицию и заявить о находке. Такие деньги просто так не терют, а что самое интересное их неприменно будут искать и поверьте мне найдут, видел подобное в жизни. Можно нарваться на бабушку конечно, которая хотела помоч внуку с покупкой квартиры, а можно на бандитов, которые будут с вами разговаривать иначе чем бабушка с милицией. Выбор за вами, есть еще конечно шанс, что это подарок с выше за котрый с вас никто не спросит, тогда лучше отдать хотябы 500 на благотворительность. дабы не спугнуть удачу!. \n',
 'Эбу в двенашке называется Итэлма что за эбу? .\tЭБУ — электронный блок управления двигателем автомобиля, его другое название — контроллер. Он при

In [ ]:
question = None
written = False

#Мы идем по всем записям, берем первую строку как вопрос
# и после знака --- находим ответ
with codecs.open("prepared_answers.txt","w", "utf-8") as fout:
    with codecs.open("Otvety.txt", "r", "utf-8") as fin:
        for line in tqdm(fin):
            if line.startswith("---"):
                written = False
                continue
            if not written and question is not None:
                fout.write(question.replace("\t", " ").strip() + "\t" + line.replace("\t", " "))
                written = True
                question = None
                continue
            if not written:
                question = line.strip()
                continue

0it [00:00, ?it/s]

Теперь нам нужно предобработать текст, чтобы обучить word2vec и получить эмбеддинги. Удаляем знаки препинания и делаем лемматизацию

In [ ]:
def preprocess_txt(line):
    spls = "".join(i for i in line.strip() if i not in exclude).split()
    spls = [morpher.parse(i.lower())[0].normal_form for i in spls]
    spls = [i for i in spls if i not in sw and i != ""]
    return spls

In [ ]:
sentences = []

morpher = MorphAnalyzer()
sw = set(get_stop_words("ru"))
exclude = set(string.punctuation)
c = 0

with codecs.open("Otvety.txt", "r", "utf-8") as fin:
    for line in tqdm(fin):
        spls = preprocess_txt(line)
        sentences.append(spls)
        c += 1
        if c > 500000:
            break

In [ ]:
# Обучим модель word2vec на наших вопросах
sentences = [i for i in sentences if len(i) > 2]
model = Word2Vec(sentences=sentences, vector_size=100, min_count=1, window=5)
model.save("w2v_model")

Теперь нам нужно сложить в индекс все вопросы. Используем библиотеку annoy. Проходимся по всем ответам, считаем, что вектор предложения - сумма word2vecов слов, которые входят в него (конечно же усредненная)

In [ ]:
index = annoy.AnnoyIndex(100 ,'angular')

index_map = {}
counter = 0

with codecs.open("prepared_answers.txt", "r", "utf-8") as f:
    for line in tqdm(f):
        n_w2v = 0
        spls = line.split("\t")
        index_map[counter] = spls[1]
        question = preprocess_txt(spls[0])
        vector = np.zeros(100)
        for word in question:
            if word in model.wv:
                vector += model.wv[word]
                n_w2v += 1
        if n_w2v > 0:
            vector = vector / n_w2v
        index.add_item(counter, vector)
            
        counter += 1

index.build(10)
index.save('speaker.ann')

Теперь остается реализовать метод, который получит на вход вопрос и найдет ответ к нему!
Мы препроцессим вопрос, находим ближайший вопрос и выбираем ответ на ближайший вопрос.

In [ ]:
def find_answer(question):
    preprocessed_question = preprocess_txt(question)
    n_w2v = 0
    vector = np.zeros(100)
    for word in preprocessed_question:
        if word in model.wv:
            vector += model.wv[word]
            n_w2v += 1
    if n_w2v > 0:
        vector = vector / n_w2v
    answer_index = index.get_nns_by_vector(vector, 1)
    return index_map[answer_index[0]]